# Introduction to Spark MLlib

Apache Spark MLlib is Apache Spark's scalable machine learning library intended for processing large-scale data. MLlib facilitates machine learning in Python, Java, Scala, and R using the Spark framework. This library is designed to interoperate with NumPy in Python and R libraries. It runs on Hadoop, Apache Mesos, Kubernetes, standalone, or in the cloud and can access diverse data sources.

## Components of Spark MLlib

Spark MLlib includes several packages for various machine learning tasks:

1. **Basic statistics** - `pyspark.ml.stat`: Provides tools for statistical analysis.
2. **Feature extraction, transformation, and selection** - `pyspark.ml.feature`: Handles preprocessing tasks like encoding, normalization, and vector assembly.
3. **Pipelines** - Integrated within `pyspark.ml`: Simplifies ML workflows.
4. **Classification and Regression** - `pyspark.ml.classification` and `pyspark.ml.regression`: For predictive modeling.
5. **Clustering** - `pyspark.ml.clustering`: For grouping a set of objects.
6. **Model selection and tuning** - Embedded within `pyspark.spark.ml`: Tools for hyperparameter tuning.
7. **Collaborative filtering** - `pyspark.ml.recommendation`: For making recommendations using algorithms like ALS.

#### Examples Extracting, Transforming, and Selecting Features
In this section, we will explore several practical examples of how to extract, transform, and select features using Spark MLlib, starting with text data vectorization through TF-IDF.


##### Feature Extractors
**TF-IDF (Term Frequency-Inverse Document Frequency)**: A feature vectorizer used primarily for text data. Here's how you can use it:

```python
from pyspark.ml.feature import HashingTF, IDF, Tokenizer

# Sample DataFrame with text
data = [("Java Python SQL", ), ("Java Scala", ), ("Python SQL", )]
df = spark.createDataFrame(data, ["text"])

# Tokenize text
tokenizer = Tokenizer(inputCol="text", outputCol="words")
wordsData = tokenizer.transform(df)

# Compute term frequency
hashingTF = HashingTF(inputCol="words", outputCol="rawFeatures")
featurizedData = hashingTF.transform(wordsData)

# Fit a IDF model
idf = IDF(inputCol="rawFeatures", outputCol="features")
idfModel = idf.fit(featurizedData)
rescaledData = idfModel.transform(featurizedData)


**StandardScaler**: Standardizes features by removing the mean and scaling to unit variance.

```python
from pyspark.ml.feature import StandardScaler

# Assuming 'df' has a column "features" that needs to be scaled
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=False)
scalerModel = scaler.fit(df)
scaledData = scalerModel.transform(df)

scaledData.show()

##### Example Feature Selectors

**VectorSlicer**: This selector extracts specific indices from a feature vector.

```python
from pyspark.ml.feature import VectorSlicer

slicer = VectorSlicer(inputCol="features", outputCols="selectedFeatures", indices=[1, 2])
slicedData = slicer.transform(df)

slicedData.show()
```

**Word2Vec**


```python
from pyspark.ml.feature import Word2Vec
from pyspark.sql.session import SparkSession

# Initialize a Spark session
spark = SparkSession.builder.appName("Word2Vec Example").getOrCreate()

# Sample data - list of words
documentDF = spark.createDataFrame([
    ("Hi I heard about Spark".split(" "), ),
    ("I wish Java could use case classes".split(" "), ),
    ("Logistic regression models are neat".split(" "), )
], ["text"])

# Create a Word2Vec model and set parameters
word2Vec = Word2Vec(vectorSize=3, minCount=0, inputCol="text", outputCol="result")

# Fit the model
model = word2Vec.fit(documentDF)

# Transform the data
result = model.transform(documentDF)

### Model Evaluation

Model evaluation is crucial in machine learning workflows to assess the performance and accuracy of predictive models. Spark MLlib provides several metrics and tools to evaluate different types of machine learning models.

#### Evaluating Classification Models

Here's an example of evaluating a Logistic Regression model using Binary Classification Evaluator.

```python
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Load training data
training = spark.createDataFrame([
    (0.0, Vectors.dense([0.5, -1.0])),
    (1.0, Vectors.dense([1.5, 1.0])),
....
], ["label", "features"])

# Train a LogisticRegression model
lr = LogisticRegression(maxIter=10, regParam=0.01)
model = lr.fit(training)
...
# Evaluate the model
evaluator = BinaryClassificationEvaluator()
accuracy = evaluator.evaluate(predictions)
print("Test Error = %g " % (1.0 - accuracy))

### ML Training with Pipeline using PySpark

In [1]:
import os

# Path to the PostgreSQL JDBC jar file
rel_path = os.path.relpath('./libs/postgresql-42.7.3.jar')

# Connection URL and properties for PostgreSQL
url = "jdbc:postgresql://fhtw-big-data.postgres.database.azure.com/music_store"
postgres_options = {
    "url": url,
    "user": "student",
    "password": "reRZ2pjg1WxqlwjU",
    "driver": "org.postgresql.Driver"
}

In [ ]:
import os
from pyspark.sql import SparkSession

# Create a Spark session and configure it to use the JDBC jar
spark = SparkSession.builder \
    .appName("PostgreSQL Integration Example") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.0") \
    .config("spark.jars", rel_path) \
    .config("spark.executor.extraClassPath", rel_path) \
    .config("spark.driver.extraClassPath", rel_path) \
    .getOrCreate()

### Data Loading and Schema Definition

Before processing the data, it is crucial to define the structure of the dataset explicitly. This is achieved by specifying a schema that describes the type of data each column holds. Defining a schema upfront helps Spark optimize data reading and processing:

- **Structured Data**: The schema corresponds to the CSV file structure, ensuring that each field is read with the correct data type.
- **Schema Definition**: Using `StructType` and `StructField`, the schema is defined to include types such as `IntegerType` and `StringType`, which instruct Spark on how to process and handle each column.
- **Data Path**: Specifies the path to the CSV file, which contains the dataset to be processed.
- **Data Reading**: Spark reads the CSV file using the defined schema and the `header=True` option, which treats the first line of the file as column names, ensuring that columns are correctly identified based on the header row in the CSV file.

This approach not only enhances performance by reducing the need for Spark to infer data types but also prevents errors due to incorrect type inference, leading to more reliable data processing pipelines.

In [3]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# Define the schema based on your CSV structure
schema = StructType([
    StructField("age", IntegerType()),
    StructField("workclass", StringType()),
    StructField("fnlwgt", IntegerType()),
    StructField("education", StringType()),
    StructField("educational-num", IntegerType()),
    StructField("marital-status", StringType()),
    StructField("occupation", StringType()),
    StructField("relationship", StringType()),
    StructField("race", StringType()),
    StructField("gender", StringType()),
    StructField("capital-gain", IntegerType()),
    StructField("capital-loss", IntegerType()),
    StructField("hours-per-week", IntegerType()),
    StructField("native-country", StringType()),
    StructField("income", StringType())
])

# Path to the dataset
data_path = "./data/adult.csv"
df = spark.read.csv(data_path, schema=schema, header=True)

In [ ]:
df.show(5)

### Feature Engineering and Pipeline Setup

Feature engineering is a critical step in the machine learning pipeline, involving the transformation of raw data into a format that is suitable for modeling. This section details the setup of feature transformers and the assembly of a machine learning pipeline in Spark:

- **Categorical and Numerical Columns**: First, we identify which columns are categorical and which are numerical. This distinction is important as it determines the type of preprocessing each column type will undergo.

- **StringIndexer for Categorical Columns**: 
    - `StringIndexer` is used to convert categorical columns into indices. This transformation is essential because many machine learning algorithms require numerical input. `handleInvalid="skip"` ensures that any invalid entries do not cause errors but are skipped instead.
    - Each categorical column is transformed to a new column with the suffix `Index`.

- **OneHotEncoder for Indexed Columns**:
    - `OneHotEncoder` takes the numeric indices output by `StringIndexer` and encodes them into a binary vector format, which is a standard way to handle categorical variables for many models. This step expands the dimensionality where each category is represented as a binary vector.
    - The outputs are named by replacing `Index` with `OHE` in the column names to signify that they have been one-hot encoded.

- **VectorAssembler**:
    - `VectorAssembler` is used to combine all the feature columns (both categorical and numerical) into a single feature vector called `features`. This feature vector is what will be fed into the machine learning algorithms.
    - It combines the one-hot encoded categorical features and the numerical features into a single vector, facilitating their use in subsequent model training.

- **Label Creation for Income Column**:
    - Another `StringIndexer` is used to convert the `income` column into a label index for classification, where Spark's MLlib expects the labels for classification tasks to be numeric.

In [5]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

# Define categorical and numerical columns based on your dataset
categorical_cols = ["workclass", "education", "marital-status", "occupation", "relationship", "race", "gender", "native-country"]
numerical_cols = ["age", "fnlwgt", "educational-num", "capital-gain", "capital-loss", "hours-per-week"]

# Indexing categorical columns
indexers = [
    StringIndexer(inputCol=c, outputCol=c+"Index", handleInvalid="skip")
    for c in categorical_cols
]

# Encoding indexed columns with OneHotEncoder
encoders = [
    OneHotEncoder(inputCols=[indexer.getOutputCol()], outputCols=[indexer.getOutputCol().replace("Index", "OHE")])
    for indexer in indexers
]

# VectorAssembler to combine categorical and numerical features into one vector
assembler = VectorAssembler(
    inputCols=[encoder.getOutputCols()[0] for encoder in encoders] + numerical_cols,
    outputCol="features"
)

# StringIndexer for the income column to be used as label
labelIndexer = StringIndexer(inputCol="income", outputCol="label", handleInvalid="skip")



### Model Definition and Pipeline Construction

In this segment of the pipeline, we define the logistic regression model and finalize the construction of the pipeline:

- **Logistic Regression Model**:
  - We instantiate a `LogisticRegression` model specifying `featuresCol` and `labelCol`. The `featuresCol` parameter points to the 'features' vector assembled in the previous step, which contains both categorical and numerical data prepared for modeling. The `labelCol` is the indexed 'income' column, which serves as the target variable for the prediction task.
  - Logistic regression is chosen for its ability to provide probabilistic outputs and effective handling of binary classification tasks.

- **Pipeline Construction**:
  - A `Pipeline` is set up to encapsulate all preprocessing steps (indexing, encoding, and assembling) along with the logistic regression model. This approach ensures that the entire sequence of transformations and the modeling step are treated as an atomic operation. This is crucial for maintaining the integrity and reproducibility of the model training process.
  - The pipeline stages include the `StringIndexers` for categorical data, `OneHotEncoders` for the indexed categorical data, the `VectorAssembler` to combine all features into a single vector, the `StringIndexer` for the target variable, and the logistic regression model itself.

By structuring the machine learning process into a pipeline, we enhance the maintainability and scalability of the model. The pipeline not only ensures that all preprocessing steps are applied consistently but also simplifies the application of the model to new data, maintaining the same preprocessing and modeling steps automatically. This setup is particularly useful in production environments where new data must be processed and predicted regularly.

In [6]:
from pyspark.ml.classification import LogisticRegression

# Logistic Regression Model
lr = LogisticRegression(featuresCol="features", labelCol="label")

# Setup the pipeline
pipeline = Pipeline(stages=indexers + encoders + [assembler, labelIndexer, lr])

### Model Training, Testing, and Evaluation

After setting up the logistic regression model and encapsulating it within a pipeline, the next crucial steps involve splitting the data, training the model, and evaluating its performance:

- **Data Splitting**:
  - The dataset is split into training and testing sets using a typical 80/20 ratio. This split ensures that we have a sufficient amount of data for training the model (`trainDF`) while reserving a portion of the dataset (`testDF`) to evaluate the model's performance on unseen data.
  - The `randomSplit` function is used with a seed to ensure reproducibility. Using a fixed seed means that the split will be the same every time the script is run, which is essential for consistent results during testing and development.

- **Model Training**:
  - The `fit` method is called on the `pipeline` with the training dataset (`trainDF`). This method applies all the transformations specified in the pipeline and uses the resulting features to train the logistic regression model.
  - During training, the pipeline processes the data by applying the transformations (indexing, encoding, assembling) and then fitting the logistic regression model on the prepared features.

- **Making Predictions**:
  - Once the model is trained, predictions can be made on new data. Here, the testing set (`testDF`) is transformed through the pipeline to ensure it undergoes the same preprocessing steps as the training set.
  - The `transform` method applies all the transformations in the pipeline to the test data and then uses the trained model to predict the outcome (`prediction`).

- **Displaying Predictions**:
  - To visually assess some of the predictions, the `show` method is used to display the first few rows of the predictions DataFrame. This output includes the features used for predictions, the true label (`label`), and the predicted label (`prediction`).
  - This step is crucial for a preliminary check of how well the model is performing and to ensure that the pipeline is functioning as expected.

This structured approach to training and testing ensures that the model is both robust and accurate, providing a reliable basis for making predictions and evaluating the model's performance on data it has never seen before.

In [ ]:
# Split the data into training and test sets
trainDF, testDF = df.randomSplit([0.8, 0.2], seed=42)

# Fit the model
model = pipeline.fit(trainDF)

# Make predictions
predictions = model.transform(testDF)

# Show some predictions
predictions.select("features", "label", "prediction").show(5)

### Model Performance Evaluation

After training the logistic regression model and making predictions on the test dataset, the next critical step is to evaluate the model's performance. This is done using the `BinaryClassificationEvaluator`, which provides a way to assess the model's accuracy and effectiveness:

- **Binary Classification Evaluator**:
  - The `BinaryClassificationEvaluator` in Spark MLlib is specifically designed for binary classification problems. It evaluates the performance of binary classifiers by comparing the predicted labels against the true labels.
  - The evaluator is configured with `rawPredictionCol` set to "rawPrediction" and `labelCol` set to "label". This configuration directs the evaluator to use the raw predictions generated by the logistic regression model and the true labels from the test data.

- **Area Under ROC**:
  - The evaluator calculates the Area Under the Receiver Operating Characteristic (ROC) curve, a common metric used to evaluate the performance of binary classification models. The ROC curve plots the true positive rate against the false positive rate at various threshold settings.
  - The Area Under the ROC (AUC) provides a single scalar value that measures the overall ability of the model to discriminate between positive and negative classes. An AUC of 1.0 represents a perfect model, while an AUC of 0.5 suggests a model that performs no better than random guessing.

- **Model Assessment**:
  - Printing the "Area under ROC" provides a clear and concise summary of the model's effectiveness in distinguishing between the classes. A higher AUC indicates a model that more effectively discriminates between those who earn more than 50K and those who do not.
  - This metric is crucial for understanding the strengths and limitations of the model, guiding potential improvements in model training, feature engineering, or even trying different modeling techniques.

This evaluation step is essential for quantitatively assessing the quality of the model based on its predictive accuracy and for making informed decisions about deploying the model in production environments or further refining it.

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Evaluate model
evaluator = BinaryClassificationEvaluator(rawPredictionCol="rawPrediction", labelCol="label")
print("Area under ROC:", evaluator.evaluate(predictions))

In [80]:
spark.stop()